In [24]:
import os
import sys
import numpy as np
import torch

# add parent directory to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.config import *
from src.data_loader import load_data
from src.preprocess import preprocess_list
from src.embedder import load_phobert, embed_texts
from src.trainer import train_svm, evaluate
from src.cso_optimizer import optimize_cso
from sklearn.model_selection import train_test_split
from src.embedding_cache import get_or_create_embeddings
from src.split_cache import get_or_create_split

In [25]:
texts, labels = load_data(DATA_PATH)

print("Samples:", len(texts))

Columns: ['title', 'label']
Loaded samples: 585
Samples: 585


In [26]:
texts = preprocess_list(texts)

In [27]:
tokenizer, model = load_phobert(PHOBERT_DIR)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1090.33it/s]


In [18]:
# X = embed_texts(texts, tokenizer, model)
# y = labels

EMBEDDING_DIR = "models/cache_embeddings"

X, y = get_or_create_embeddings(
    texts=texts,
    labels=labels,
    embed_fn=embed_texts,
    tokenizer=tokenizer,
    model=model,
    save_dir=EMBEDDING_DIR
)

⚡ Creating embeddings... (first time only)
✅ Saved embeddings:
   X: (585, 768)
   y: (585,)


In [23]:
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y,
#     test_size=TEST_SIZE,
#     stratify=y,
#     random_state=RANDOM_STATE
# )

SPLIT_DIR = "models/cache_split"

X_train, X_test, y_train, y_test = get_or_create_split(
    X, y,
    save_dir=SPLIT_DIR
)

⚡ Creating train/test split...
✅ Saved split:
Train: (468, 768)
Test : (117, 768)


In [28]:
clf = train_svm(X_train, y_train)
evaluate(clf, X_test, y_test)

Accuracy: 0.5641025641025641
F1: 0.5552786566718455
              precision    recall  f1-score   support

           0       0.80      0.89      0.84         9
           1       0.57      0.44      0.50         9
           2       0.38      0.33      0.35         9
           3       0.33      0.44      0.38         9
           4       0.70      0.78      0.74         9
           5       0.50      0.56      0.53         9
           6       0.70      0.78      0.74         9
           7       0.67      0.89      0.76         9
           8       0.71      0.56      0.62         9
           9       0.38      0.33      0.35         9
          10       0.29      0.22      0.25         9
          11       0.78      0.78      0.78         9
          12       0.43      0.33      0.38         9

    accuracy                           0.56       117
   macro avg       0.56      0.56      0.56       117
weighted avg       0.56      0.56      0.56       117



In [29]:
best_C = optimize_cso(X_train, y_train)
print("Best C:", best_C)

C=1.0269 -> 0.6073
C=0.0390 -> 0.5605
C=0.0113 -> 0.5323
C=1091.3469 -> 0.5592
C=7549.6589 -> 0.5592
C=7.4966 -> 0.5803
C=0.0388 -> 0.5605
C=0.0114 -> 0.5323
C=1118.8398 -> 0.5592
C=7486.7845 -> 0.5592
C=7.3172 -> 0.5803
Iter 0 | Best fitness: -0.6073
C=0.0431 -> 0.5648
C=0.0112 -> 0.5323
C=1072.8966 -> 0.5592
C=9070.5202 -> 0.5592
C=7.1719 -> 0.5803
Iter 1 | Best fitness: -0.6073
C=0.0471 -> 0.5640
C=0.0119 -> 0.5344
C=1058.4041 -> 0.5592
C=7312.0807 -> 0.5592
C=7.2859 -> 0.5803
C=102718.8889 -> 0.5592
Iter 2 | Best fitness: -0.6073

Best solution: [0.01153464]
Best fitness: -0.6072574613870161
Best C: 1.0269153455837334


In [30]:
best_model = train_svm(X_train, y_train, C=best_C)
evaluate(best_model, X_test, y_test)

Accuracy: 0.5641025641025641
F1: 0.5552786566718455
              precision    recall  f1-score   support

           0       0.80      0.89      0.84         9
           1       0.57      0.44      0.50         9
           2       0.38      0.33      0.35         9
           3       0.33      0.44      0.38         9
           4       0.70      0.78      0.74         9
           5       0.50      0.56      0.53         9
           6       0.70      0.78      0.74         9
           7       0.67      0.89      0.76         9
           8       0.71      0.56      0.62         9
           9       0.38      0.33      0.35         9
          10       0.29      0.22      0.25         9
          11       0.78      0.78      0.78         9
          12       0.43      0.33      0.38         9

    accuracy                           0.56       117
   macro avg       0.56      0.56      0.56       117
weighted avg       0.56      0.56      0.56       117

